In [35]:
!pip install torch_geometric
!pip install trimesh
!pip install rtree
!apt-get install -y libspatialindex-dev
print("✅ Librairies installées.")
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Pour la 3D et les graphes (selon ton projet)
import torch_geometric
from torch_geometric.data import Data, Batch, DataLoader
import sys
import os
import shutil
import trimesh
from google.colab import drive

# Configuration du Device (GPU est fortement recommandé pour 54k features)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device utilisé : {device}")

# Fixer les graines aléatoires pour la reproductibilité
torch.manual_seed(42)
np.random.seed(42)

# Création du dossier pour sauvegarder les modèles et résultats
os.makedirs("gan_checkpoints", exist_ok=True)
os.makedirs("generated_meshes", exist_ok=True)
print("✅ Dossiers de sortie créés.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libspatialindex-dev is already the newest version (1.9.3-2).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
✅ Librairies installées.
✅ Device utilisé : cuda
✅ Dossiers de sortie créés.


In [36]:

from torch_geometric.loader import DataLoader

# 1. Montage du Drive
drive.mount('/content/drive', force_remount=True)

# 2. Configuration des chemins (Mise à jour avec le bon nom de dossier)
project_path = '/content/drive/MyDrive/Data_challenge_Drive' # Attention majuscules/minuscules
data_root = os.path.join(project_path, 'data')

# Ajout du path système
if project_path not in sys.path:
    sys.path.append(project_path)

# 3. Structure de fichiers
raw_dir = os.path.join(data_root, 'raw')
os.makedirs(raw_dir, exist_ok=True)

if os.path.exists(data_root):
    for file in os.listdir(data_root):
        if file.endswith('.obj'):
            # On déplace uniquement si ce n'est pas déjà dans raw
            if not os.path.exists(os.path.join(raw_dir, file)):
                shutil.move(os.path.join(data_root, file), os.path.join(raw_dir, file))

# 4. Import du module
try:
    import trimesh
    import Data_loader_final as loader_utils
    print("✅ Module Data_loader_final chargé.")
except ImportError as e:
    raise RuntimeError(f"❌ Erreur import : {e}. Vérifie le nom du dossier et l'installation de trimesh.")

# 5. DÉFINITION DE LA CLASSE (Avec Correctif 'pos')
class FemurDataset(Dataset):
    def __init__(self, root, mode='graph'):
        self.root = root
        load_path = os.path.join(root, 'raw') if os.path.exists(os.path.join(root, 'raw')) else root

        print(f"📂 Chargement des meshes depuis : {load_path}")

        # Chargement via tes outils
        self.meshes = loader_utils.load_meshes(load_path, normalize=True)
        self.data_list = loader_utils.meshes_to_data(self.meshes, mode=mode)

        # --- CORRECTIF CRITIQUE ---
        # Ton script met les sommets dans 'x'. On les duplique dans 'pos'
        # pour que PyTorch Geometric et le VAE puissent les trouver facilement.
        for data in self.data_list:
            if not hasattr(data, 'pos') or data.pos is None:
                data.pos = data.x
        # --------------------------

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

# 6. Instanciation
BATCH_SIZE = 4
print("⏳ Création du Dataset...")
try:
    dataset = FemurDataset(root=data_root, mode='graph')
except Exception as e:
    raise RuntimeError(f"❌ Erreur création Dataset : {e}")

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# 7. Vérifications
print(f"\n📊 Résumé :")
print(f"   - Nombre de meshes chargés : {len(dataset)}")

if len(dataset) > 0:
    sample = dataset[0]
    # Maintenant sample.pos existe grâce au correctif !
    NUM_VERTICES = sample.pos.shape[0]
    INPUT_DIM = NUM_VERTICES * 3

    print(f"   - Vertices par mesh : {NUM_VERTICES}")
    print(f"   - Input Dimension : {INPUT_DIM}")
else:
    print("⚠️ Dataset vide.")

Mounted at /content/drive
✅ Module Data_loader_final chargé.
⏳ Création du Dataset...
📂 Chargement des meshes depuis : /content/drive/MyDrive/Data_challenge_Drive/data/raw
Loaded 24 meshes from /content/drive/MyDrive/Data_challenge_Drive/data/raw

📊 Résumé :
   - Nombre de meshes chargés : 24
   - Vertices par mesh : 18291
   - Input Dimension : 54873


In [37]:
def prepare_batch_for_vae(data_batch):
    """
    Transforme un batch PyG (Graphes) en tenseur plat (Batch, 54000).
    """
    # Vérification que 'pos' existe (au cas où)
    if not hasattr(data_batch, 'pos') or data_batch.pos is None:
         # Fallback sur 'x' si 'pos' est manquant
        input_data = data_batch.x
    else:
        input_data = data_batch.pos

    batch_size = data_batch.num_graphs

    # Vérification de sécurité dimensionnelle
    if input_data.shape[0] % batch_size != 0:
        raise ValueError(f"Erreur dimensions : {input_data.shape[0]} points ne sont pas divisibles par batch_size={batch_size}.")

    flat_tensor = input_data.view(batch_size, -1)
    return flat_tensor.to(device)

def save_mesh_obj(vertices_flat, filename, faces_ref=None):
    """
    Sauvegarde un mesh généré au format .obj
    Gère automatiquement l'orientation des faces (N,3) ou (3,N).
    """
    # 1. Reshape des sommets (Flatten -> N, 3)
    vertices = vertices_flat.detach().cpu().numpy().reshape(-1, 3)

    with open(filename, 'w') as f:
        f.write("# Mesh généré par VAE-GAN - Projet Fémur\n")

        # 2. Écriture des sommets
        for v in vertices:
            f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")

        # 3. Écriture des faces (Topologie)
        if faces_ref is not None:
            faces_np = faces_ref.detach().cpu().numpy()

            # --- DÉTECTION AUTOMATIQUE DE L'ORIENTATION ---
            # Trimesh donne (N, 3), mais PyG utilise parfois (2, E) ou (3, N)
            # On veut le format (N_faces, 3) pour l'écriture
            if faces_np.shape[0] == 3 and faces_np.shape[1] > 3:
                # Si c'est (3, N), on transpose
                faces_np = faces_np.T

            # Écriture (OBJ commence l'indexation à 1, pas 0)
            for face in faces_np:
                f.write(f"f {int(face[0])+1} {int(face[1])+1} {int(face[2])+1}\n")

    print(f"💾 Mesh sauvegardé : {filename}")

In [38]:
# Modèle VAE

class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=128): # On retire la valeur par défaut 54000 pour forcer la bonne valeur
        super(VAE, self).__init__()

        # Encoder : Compression
        # 54k -> 2048 -> 512
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.BatchNorm1d(2048),
            nn.LeakyReLU(0.2),
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2)
        )

        # Espace Latent (Moyenne et Variance)
        self.fc_mu = nn.Linear(512, latent_dim)
        self.fc_logvar = nn.Linear(512, latent_dim)

        # Decoder : Reconstruction
        # 128 -> 512 -> 2048 -> 54k
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 2048),
            nn.BatchNorm1d(2048),
            nn.LeakyReLU(0.2),
            nn.Linear(2048, input_dim)
            # Pas de Sigmoid/Tanh ici car tes coordonnées normalisées peuvent être négatives ou > 1
        )

    def reparameterize(self, mu, logvar):
        """L'astuce de reparamétrisation pour permettre la backpropagation"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decoder(z)
        return reconstruction, mu, logvar

print("✅ Classe VAE définie.")

✅ Classe VAE définie.


In [39]:
# Modèle GAN

class LatentGenerator(nn.Module):
    def __init__(self, latent_dim=128, noise_dim=100):
        super(LatentGenerator, self).__init__()
        # Le générateur part du bruit pour recréer un code latent plausible
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, latent_dim)
            # Sortie = vecteur de taille 128 (compatible avec le decodeur VAE)
        )

    def forward(self, z):
        return self.model(z)

class LatentDiscriminator(nn.Module):
    def __init__(self, latent_dim=128):
        super(LatentDiscriminator, self).__init__()
        # Le discriminateur juge si un code latent est réel (issu du VAE) ou faux (généré)
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),  #  AJOUT SÉCURITÉ : Empêche l'overfitting sur peu de données

            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),  #  AJOUT SÉCURITÉ

            nn.Linear(128, 1),
            nn.Sigmoid()      # Sortie entre 0 (Fake) et 1 (Real)
        )

    def forward(self, z):
        return self.model(z)

print("✅ Classes GAN définies (avec Dropout pour la stabilité).")

✅ Classes GAN définies (avec Dropout pour la stabilité).


In [40]:
# INSTANCIATION

# Vérification de sécurité avant de commencer
if 'INPUT_DIM' not in globals():
    raise ValueError("❌ La variable INPUT_DIM n'est pas définie. Assure-toi d'avoir exécuté la Cellule 2 (Chargement Données) avec succès.")

# Hyperparamètres
LATENT_DIM = 128
NOISE_DIM = 100

# Learning Rates (Tu pourras les ajuster si l'entraînement est trop lent/instable)
LR_VAE = 1e-4
LR_GAN = 1e-4

# Poids de la perte KL (Beta-VAE)
# - Si tes recontructions sont floues -> Diminue BETA (ex: 0.001)
# - Si l'espace latent est trop dispersé -> Augmente BETA (ex: 0.1)
BETA = 0.005

# Instanciation des modèles
print(f"🏗️ Création des modèles avec Input Dim: {INPUT_DIM}...")
vae = VAE(input_dim=INPUT_DIM, latent_dim=LATENT_DIM).to(device)
generator = LatentGenerator(latent_dim=LATENT_DIM, noise_dim=NOISE_DIM).to(device)
discriminator = LatentDiscriminator(latent_dim=LATENT_DIM).to(device)

# Optimiseurs
opt_vae = optim.Adam(vae.parameters(), lr=LR_VAE)
opt_G = optim.Adam(generator.parameters(), lr=LR_GAN)
opt_D = optim.Adam(discriminator.parameters(), lr=LR_GAN)

# Fonctions de perte
criterion_mse = nn.MSELoss()  # Pour la reconstruction VAE
criterion_bce = nn.BCELoss()  # Pour le duel Real/Fake du GAN

# Calcul du nombre de paramètres (Info utile pour le rapport)
vae_params = sum(p.numel() for p in vae.parameters() if p.requires_grad)
gan_params = sum(p.numel() for p in generator.parameters()) + sum(p.numel() for p in discriminator.parameters())

print(f"✅ Modèles initialisés sur {device}.")
print(f"   - Paramètres VAE : {vae_params:,}")
print(f"   - Paramètres GAN : {gan_params:,}")

🏗️ Création des modèles avec Input Dim: 54873...
✅ Modèles initialisés sur cuda.
   - Paramètres VAE : 227,124,057
   - Paramètres GAN : 290,689


In [41]:
# FONCTION DE LOSS DE LISSAGE

def compute_smoothness_loss(pred_mesh_tensor, edge_index, batch_size):
    """
    Calcule une perte qui pénalise les surfaces rugueuses (Laplacian Smoothing).
    pred_mesh_tensor: (Batch, 54000) - Les sommets aplatis sortis du modèle
    edge_index: Les connexions du graphe (Tenseur [2, Num_Edges])
    """
    # 1. On remet les données en format (Total_Points, 3) pour le graphe
    # (Batch * 18000, 3)
    pred_verts = pred_mesh_tensor.view(-1, 3)

    # 2. On récupère les positions des voisins (Source -> Target)
    src_idx, dst_idx = edge_index

    # 3. Calcul de la distance moyenne entre voisins (Edge Length Regularization)
    # C'est une version simplifiée et très rapide du Laplacien qui marche bien pour les GANs
    # On veut que la distance entre deux points connectés soit petite et uniforme

    # Positions des points sources et destinations
    p_src = pred_verts[src_idx]
    p_dst = pred_verts[dst_idx]

    # Distance au carré entre voisins
    edge_len_sq = (p_src - p_dst).pow(2).sum(dim=-1)

    # La loss est la moyenne de ces distances (pousse à contracter les spikes)
    loss_smooth = edge_len_sq.mean()

    return loss_smooth

print("✅ Fonction Smoothness Loss définie.")

✅ Fonction Smoothness Loss définie.


In [42]:
# ENTRAÎNEMENT STANDARD

# Paramètres
EPOCHS = 200
WARMUP_EPOCHS = 50      # Pendant 50 époques, on n'entraîne QUE le VAE (stabilité)
CLIP_VALUE = 1.0
LAMBDA_SMOOTH = 15.0    # Force de lissage élevée pour éviter les spikes
CHECKPOINT_DIR = "gan_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"🚀 Démarrage : Warm-up de {WARMUP_EPOCHS} époques avec Lambda={LAMBDA_SMOOTH}")

history = {'vae': [], 'd': [], 'g': [], 'smooth': []}

for epoch in range(EPOCHS):
    # Mode entraînement
    vae.train()
    generator.train()
    discriminator.train()

    # Accumulateurs pour les moyennes
    vae_loss_acc = 0
    smooth_loss_acc = 0
    d_loss_acc = 0
    g_loss_acc = 0
    batches = 0

    is_warmup = (epoch < WARMUP_EPOCHS)

    for batch in train_loader:
        batches += 1
        real_meshes = prepare_batch_for_vae(batch) # (Batch, N*3)
        curr_batch_size = real_meshes.size(0)
        batch_edge_index = batch.edge_index.to(device)

        # =======================
        # 1. TRAIN VAE
        # =======================
        opt_vae.zero_grad()

        # Reconstruction directe (Méthode Classique)
        recon, mu, logvar = vae(real_meshes)

        # Loss Reconstruction
        loss_recon = criterion_mse(recon, real_meshes)

        # KL Divergence
        beta_current = 0.0001 if is_warmup else BETA
        loss_kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / curr_batch_size

        # Smoothness (Anti-Spikes)
        loss_smooth = compute_smoothness_loss(recon, batch_edge_index, curr_batch_size)

        # Total Loss
        loss_vae = loss_recon + (beta_current * loss_kl) + (LAMBDA_SMOOTH * loss_smooth)

        loss_vae.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), CLIP_VALUE)
        opt_vae.step()

        vae_loss_acc += loss_recon.item()
        smooth_loss_acc += loss_smooth.item()

        # =======================
        # 2. TRAIN GAN
        # =======================
        if not is_warmup:
            # --- A. DISCRIMINATOR ---
            # Le discriminateur apprend à distinguer les Latents Vrais (mu) des Faux
            with torch.no_grad():
                _, mu_real, _ = vae(real_meshes)

            z_noise = torch.randn(curr_batch_size, NOISE_DIM).to(device)
            fake_latents = generator(z_noise)

            opt_D.zero_grad()
            pred_real = discriminator(mu_real)
            loss_d_real = criterion_bce(pred_real, torch.ones_like(pred_real))

            pred_fake = discriminator(fake_latents.detach())
            loss_d_fake = criterion_bce(pred_fake, torch.zeros_like(pred_fake))

            loss_d = (loss_d_real + loss_d_fake) / 2
            loss_d.backward()
            opt_D.step()
            d_loss_acc += loss_d.item()

            # --- B. GENERATOR ---
            # Le générateur essaie de tromper le discriminateur
            opt_G.zero_grad()
            pred_fake_g = discriminator(fake_latents)
            loss_g = criterion_bce(pred_fake_g, torch.ones_like(pred_fake_g))
            loss_g.backward()
            opt_G.step()
            g_loss_acc += loss_g.item()
        else:
            d_loss_acc += 0.0
            g_loss_acc += 0.0

    # --- LOGS ET AFFICHAGE ---
    avg_vae = vae_loss_acc / batches
    avg_smooth = smooth_loss_acc / batches
    avg_d = d_loss_acc / batches
    avg_g = g_loss_acc / batches

    # Stockage pour graphiques éventuels
    history['vae'].append(avg_vae)
    history['smooth'].append(avg_smooth)
    history['d'].append(avg_d)
    history['g'].append(avg_g)

    status = "🔥 WARMUP" if is_warmup else "⚔️ GAN TRAIN"

    if (epoch + 1) % 5 == 0:
        # l'affichage des pertes GAN (D et G)
        print(f"Ep {epoch+1:03d} [{status}] | Smooth: {avg_smooth:.4f} | VAE: {avg_vae:.4f} | D Loss: {avg_d:.4f} | G Loss: {avg_g:.4f}")

    # Sauvegarde régulière
    if (epoch + 1) % 50 == 0:
        torch.save(generator.state_dict(), os.path.join(CHECKPOINT_DIR, "generator_classic.pth"))
        torch.save(vae.state_dict(), os.path.join(CHECKPOINT_DIR, "vae_classic.pth"))

print("🏁 Entraînement terminé.")

🚀 Démarrage : Warm-up de 50 époques avec Lambda=15.0
Ep 005 [🔥 WARMUP] | Smooth: 0.5475 | VAE: 0.2235 | D Loss: 0.0000 | G Loss: 0.0000
Ep 010 [🔥 WARMUP] | Smooth: 0.2646 | VAE: 0.1576 | D Loss: 0.0000 | G Loss: 0.0000
Ep 015 [🔥 WARMUP] | Smooth: 0.0845 | VAE: 0.1026 | D Loss: 0.0000 | G Loss: 0.0000
Ep 020 [🔥 WARMUP] | Smooth: 0.0325 | VAE: 0.0548 | D Loss: 0.0000 | G Loss: 0.0000
Ep 025 [🔥 WARMUP] | Smooth: 0.0257 | VAE: 0.0254 | D Loss: 0.0000 | G Loss: 0.0000
Ep 030 [🔥 WARMUP] | Smooth: 0.0112 | VAE: 0.0171 | D Loss: 0.0000 | G Loss: 0.0000
Ep 035 [🔥 WARMUP] | Smooth: 0.0137 | VAE: 0.0131 | D Loss: 0.0000 | G Loss: 0.0000
Ep 040 [🔥 WARMUP] | Smooth: 0.0082 | VAE: 0.0158 | D Loss: 0.0000 | G Loss: 0.0000
Ep 045 [🔥 WARMUP] | Smooth: 0.0041 | VAE: 0.0142 | D Loss: 0.0000 | G Loss: 0.0000
Ep 050 [🔥 WARMUP] | Smooth: 0.0040 | VAE: 0.0135 | D Loss: 0.0000 | G Loss: 0.0000
Ep 055 [⚔️ GAN TRAIN] | Smooth: 0.0189 | VAE: 0.0144 | D Loss: 0.6889 | G Loss: 0.5878
Ep 060 [⚔️ GAN TRAIN] | Smooth

In [46]:
# --- CELLULE 8 : PRODUCTION DE MASSE (CORRIGÉE CLASSIC) ---
import trimesh
import os
import torch
import numpy as np

# Dossier de sortie
output_dir = os.path.join(project_path, 'resultats_gan_classique')
os.makedirs(output_dir, exist_ok=True)

def generate_femurs(nb_samples=10):
    print(f"🏭 Démarrage de la production de {nb_samples} fémurs...")

    generator.eval()
    vae.eval()

    # Récupération de la topologie (faces) pour reconstruire le maillage
    ref_data = dataset[0]
    if hasattr(ref_data, 'faces'): faces = ref_data.faces
    elif hasattr(ref_data, 'face'): faces = ref_data.face
    else: faces = None

    if faces is not None and faces.shape[0] == 3:
        faces_np = faces.t().cpu().numpy()
    else:
        faces_np = faces.cpu().numpy()

    for i in range(nb_samples):
        with torch.no_grad():
            # 1. Génération Latente (Depuis le bruit)
            z_noise = torch.randn(1, NOISE_DIM).to(device)
            generated_latent = generator(z_noise)

            # 2. Décodage VAE (Direct - Pas de résidus ici !)
            generated_mesh_tensor = vae.decoder(generated_latent)

            # Conversion Numpy
            verts = generated_mesh_tensor.view(-1, 3).cpu().numpy()

        # 3. Création du Mesh
        mesh = trimesh.Trimesh(vertices=verts, faces=faces_np, process=False)

        # 4. Lissage Laplacien (Correctif)
        # 20 itérations = Lissage propre sans effet "pôle nord" ni perte excessive de volume
        trimesh.smoothing.filter_laplacian(mesh, iterations=50)

        # 5. Sauvegarde
        filename = f"femur_classic_{i+1:03d}.obj"
        save_path = os.path.join(output_dir, filename)

        # Include normals pour un meilleur rendu
        mesh.export(save_path, include_normals=True)
        print(f"   ✅ Fémur #{i+1} généré -> {filename}")

    print(f"\n📁 Fichiers sauvegardés dans : {output_dir}")

# Lancer la génération
generate_femurs(nb_samples=10)

🏭 Démarrage de la production de 10 fémurs...
   ✅ Fémur #1 généré -> femur_classic_001.obj
   ✅ Fémur #2 généré -> femur_classic_002.obj
   ✅ Fémur #3 généré -> femur_classic_003.obj
   ✅ Fémur #4 généré -> femur_classic_004.obj
   ✅ Fémur #5 généré -> femur_classic_005.obj
   ✅ Fémur #6 généré -> femur_classic_006.obj
   ✅ Fémur #7 généré -> femur_classic_007.obj
   ✅ Fémur #8 généré -> femur_classic_008.obj
   ✅ Fémur #9 généré -> femur_classic_009.obj
   ✅ Fémur #10 généré -> femur_classic_010.obj

📁 Fichiers sauvegardés dans : /content/drive/MyDrive/Data_challenge_Drive/resultats_gan_classique


In [47]:
# VISUALISATION (POUR RAPPORT)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def generate_figure_3_gallery():
    # 1. Récupération des fichiers du bon dossier
    output_dir = os.path.join(project_path, 'resultats_gan_classique')

    if not os.path.exists(output_dir):
        print(f"⚠️ Dossier {output_dir} introuvable. Lancez la Cellule 8 d'abord !")
        return

    files = sorted([os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.obj')])

    if len(files) < 3:
        print(f"⚠️ Pas assez de fichiers ({len(files)}). Lancez la Cellule 8 pour en générer au moins 3.")
        files_to_show = files
    else:
        files_to_show = files[:3] # On prend les 3 premiers

    print(f"📸 Visualisation de : {[os.path.basename(f) for f in files_to_show]}")

    # 2. Création de la figure (3 colonnes)
    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=("Échantillon A", "Échantillon B", "Échantillon C")
    )

    # 3. Remplissage
    for index, file_path in enumerate(files_to_show):
        mesh = trimesh.load(file_path, process=False)

        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(trimesh.Trimesh(vertices=g.vertices, faces=g.faces)
                                                for g in mesh.geometry.values()))

        x, y, z = mesh.vertices.T
        i, j, k = mesh.faces.T

        fig.add_trace(
            go.Mesh3d(
                x=x, y=y, z=z,
                i=i, j=j, k=k,
                color='#d4c5a3', # Couleur os
                opacity=1.0,
                name=f'Femur {index+1}',
                lighting=dict(ambient=0.4, diffuse=0.5, roughness=0.1, specular=0.3),
                lightposition=dict(x=100, y=200, z=150)
            ),
            row=1, col=index+1
        )

    # 4. Style
    axis_config = dict(visible=False, showgrid=False, showbackground=False)
    layout_dict = dict(
        title_text="Figure 3 : Diversité des Fémurs Générés",
        height=500, width=1200,
        showlegend=False
    )
    # Appliquer le style sans axe à tous les subplots
    layout_dict['scene'] = dict(xaxis=axis_config, yaxis=axis_config, zaxis=axis_config, aspectmode='data')
    layout_dict['scene2'] = dict(xaxis=axis_config, yaxis=axis_config, zaxis=axis_config, aspectmode='data')
    layout_dict['scene3'] = dict(xaxis=axis_config, yaxis=axis_config, zaxis=axis_config, aspectmode='data')

    fig.update_layout(**layout_dict)
    fig.show()

generate_figure_3_gallery()

📸 Visualisation de : ['femur_classic_001.obj', 'femur_classic_002.obj', 'femur_classic_003.obj']


In [45]:
# HEATMAP D'ERREUR

def visualize_reconstruction_error_classic():
    print("🔬 Calcul de l'erreur de reconstruction (Classique)...")

    # 1. Prendre un VRAI fémur
    dataset_sample = dataset[0]
    real_pos = dataset_sample.pos.to(device)
    real_batch = real_pos.view(1, -1)

    # 2. Reconstruction VAE (Directe)
    vae.eval()
    with torch.no_grad():
        recon_batch, _, _ = vae(real_batch)

    # 3. Post-Traitement (Même lissage que la production pour être juste)
    recon_verts_np = recon_batch.view(-1, 3).cpu().numpy()

    if hasattr(dataset_sample, 'faces'): faces = dataset_sample.faces
    elif hasattr(dataset_sample, 'face'): faces = dataset_sample.face
    else: faces = None
    if faces is not None and faces.shape[0] == 3: faces = faces.t()

    mesh = trimesh.Trimesh(vertices=recon_verts_np, faces=faces.cpu().numpy(), process=False)
    trimesh.smoothing.filter_laplacian(mesh, iterations=50)
    smoothed_pos = mesh.vertices

    # 4. Calcul Erreur
    real_pos_np = real_pos.cpu().numpy()
    diff = real_pos_np - smoothed_pos
    dist = np.linalg.norm(diff, axis=1)

    print("-" * 30)
    print("📋 RÉSULTATS POUR LE RAPPORT :")
    print(f"Metric        | Value (mm)")
    print(f"--------------|-----------")
    print(f"Mean Error    | {np.mean(dist):.4f}")
    print(f"Max Error     | {np.max(dist):.4f}")
    print(f"Median Error  | {np.median(dist):.4f}")
    print("-" * 30)

    # 5. Affichage
    x, y, z = smoothed_pos.T
    i, j, k = faces.cpu().numpy().T

    fig = go.Figure(data=[
        go.Mesh3d(
            x=x, y=y, z=z,
            i=i, j=j, k=k,
            intensity=dist,
            colorscale='Jet',
            cmin=0, cmax=np.percentile(dist, 95),
            showscale=True,
            name='Erreur'
        )
    ])
    fig.update_layout(title="Heatmap d'Erreur (Reconstruction Classique)", scene=dict(aspectmode='data'))
    fig.show()

visualize_reconstruction_error_classic()

🔬 Calcul de l'erreur de reconstruction (Classique)...
------------------------------
📋 RÉSULTATS POUR LE RAPPORT :
Metric        | Value (mm)
--------------|-----------
Mean Error    | 0.2121
Max Error     | 0.3724
Median Error  | 0.2295
------------------------------
